# patches

> Cut full-resolution image/mask pairs into overlapping training patches. Generic over the number of source channels and the number of mask classes.

The disk layout is modelled on the source layout: when there is a single source channel, image and mask patches share the same base name (`{id}.png` in `images/` and `masks/` respectively). For multi-channel sources each channel is saved as `{id}_ch{i}.png`. The model input image is assembled later (see `channels`), so the same patches feed binary or multi-class tasks.

In [ ]:
#| default_exp patches

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import cv2
import numpy as np

from segmentation_tools_pt.masks import binarize, fit_rotated_rects_multiclass

## Patch configuration

One dataclass holds every knob for patch extraction. Overlap (`stride < patch_size`) ensures no target sits permanently on a hard patch boundary.

In [ ]:
#| export
@dataclass
class PatchConfig:
    "Knobs controlling how full images are cut into patches."
    patch_size: int = 400          # side length of a saved patch (px)
    stride: int = 200              # step between patch centres (< patch_size -> overlap)
    min_fg_pixels: int = 500       # patches below this foreground count are 'empty'
    val_fraction: float = 0.2      # fraction of SOURCE images held out for validation
    max_empty_fraction: float = 0.15  # probability of keeping an empty patch
    seed: int = 42                 # base RNG seed

## Grid of patch centres

First job: given an image size, where do the patch centres go? We step by `stride` and always add a final anchor flush with the right/bottom edge so the border is covered.

In [ ]:
# experiment
h, w, patch, stride = 1000, 1200, 400, 200
half = patch // 2
ys = list(range(half, h - half + 1, stride))
if ys[-1] < h - half: ys.append(h - half)
print('row centres:', ys)

In [ ]:
#| export
def grid_centers(
    h: int, w: int,        # image height and width
    patch_size: int,       # patch side length
    stride: int,           # step between centres
) -> List[Tuple[int, int]]:
    """Return `(cy, cx)` patch centres tiling an `h`x`w` image, edges included.

    Example:
        >>> grid_centers(8, 8, 4, 4)
        [(2, 2), (2, 6), (6, 2), (6, 6)]
    """
    half = patch_size // 2
    ys = list(range(half, h - half + 1, stride)) or [half]
    xs = list(range(half, w - half + 1, stride)) or [half]
    if ys[-1] < h - half: ys.append(h - half)
    if xs[-1] < w - half: xs.append(w - half)
    return [(cy, cx) for cy in ys for cx in xs]

In [ ]:
# test
c = grid_centers(8, 8, 4, 4)
assert c == [(2, 2), (2, 6), (6, 2), (6, 6)]
# overlap case still covers the far edge
assert grid_centers(10, 10, 4, 4)[-1] == (8, 8)

## Crop one patch

Second job: extract a square crop centred at `(cy, cx)`. Works for 2-D (mask) and 3-D (multi-channel) arrays.

In [ ]:
#| export
def crop_patch(
    arr: np.ndarray,       # (H, W) or (H, W, C) array
    cy: int, cx: int,      # patch centre
    size: int,             # patch side length
) -> np.ndarray:
    """Crop a `size`x`size` window centred at `(cy, cx)`.

    Example:
        >>> crop_patch(np.arange(64).reshape(8, 8), 2, 2, 4).shape
        (4, 4)
    """
    half = size // 2
    y0, x0 = cy - half, cx - half
    return arr[y0:y0 + size, x0:x0 + size]

In [ ]:
# test
assert crop_patch(np.arange(64).reshape(8, 8), 2, 2, 4).shape == (4, 4)
assert crop_patch(np.zeros((8, 8, 3)), 4, 4, 4).shape == (4, 4, 3)

## Per-class pixel counts

Third job: how much of each class is in a mask patch? Used to flag empty patches and to weight sampling later.

In [ ]:
#| export
def count_class_pixels(
    mask: np.ndarray,   # (H, W) integer-label mask
    num_classes: int,   # number of foreground classes K
) -> Dict[int, int]:
    """Count pixels per foreground label `1..K`.

    Example:
        >>> count_class_pixels(np.array([[0, 1], [2, 2]]), 2)
        {1: 1, 2: 2}
    """
    return {c: int((mask == c).sum()) for c in range(1, num_classes + 1)}

In [ ]:
# test
assert count_class_pixels(np.array([[0, 1], [2, 2]]), 2) == {1: 1, 2: 2}

## Extract all patches from one sample

The composing function. It reads the source channels and mask, optionally snaps mask blobs to rectangles, tiles the grid, keeps every patch with enough foreground (and a sampled fraction of empty ones), writes them out, and returns per-patch metadata. Reproducible per sample via a seed derived from `(seed, sample_id)`.

In [ ]:
#| export
def extract_patches_from_sample(
    sample_id: str,                  # unique id for the source image
    channel_paths: Sequence[str],    # paths to the source single-channel images
    mask_path: str,                  # path to the integer-label mask
    output_dir: Path,                # split dir; gets images/ and masks/ subdirs
    cfg: PatchConfig,                # extraction settings
    num_classes: int = 1,            # number of foreground classes
    fit_rects: bool = False,         # snap mask blobs to rotated rectangles first
) -> List[Dict]:
    """Tile one source image into patches on disk and return their metadata.

    For a single source channel the image patch is saved as `{pid}.png` so that
    image and mask filenames match. Multi-channel sources still use `{pid}_ch{i}.png`.
    The mask is always saved as `{pid}.png`.
    """
    output_dir = Path(output_dir)
    (output_dir / 'images').mkdir(parents=True, exist_ok=True)
    (output_dir / 'masks').mkdir(parents=True, exist_ok=True)

    channels = [cv2.imread(str(p), cv2.IMREAD_GRAYSCALE) for p in channel_paths]
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    for arr, p in zip(channels, channel_paths):
        assert arr is not None, f'Cannot read channel: {p}'
    assert mask is not None, f'Cannot read mask: {mask_path}'

    mask = binarize(mask) if num_classes == 1 else mask.astype(np.int32)
    if fit_rects:
        mask = fit_rotated_rects_multiclass(mask, max(num_classes, 1))

    # Deterministic per-sample RNG: same run -> same patches, different sources differ.
    rng = np.random.default_rng(abs(hash((cfg.seed, sample_id))) % (2 ** 32))
    h, w = channels[0].shape
    single_channel = len(channels) == 1
    meta: List[Dict] = []
    for cy, cx in grid_centers(h, w, cfg.patch_size, cfg.stride):
        mpatch = crop_patch(mask, cy, cx, cfg.patch_size)
        fg = int((mpatch > 0).sum())
        is_empty = fg < cfg.min_fg_pixels
        if is_empty and rng.random() > cfg.max_empty_fraction:
            continue
        pid = f'{sample_id}_y{cy:05d}_x{cx:05d}'
        if single_channel:
            cv2.imwrite(str(output_dir / 'images' / f'{pid}.png'), crop_patch(channels[0], cy, cx, cfg.patch_size))
        else:
            for i, ch in enumerate(channels):
                cv2.imwrite(str(output_dir / 'images' / f'{pid}_ch{i}.png'), crop_patch(ch, cy, cx, cfg.patch_size))
        out_mask = (mpatch * 255).astype(np.uint8) if num_classes == 1 else mpatch.astype(np.uint8)
        cv2.imwrite(str(output_dir / 'masks' / f'{pid}.png'), out_mask)
        meta.append({
            'sample_id': pid, 'source_id': sample_id,
            'center_y': cy, 'center_x': cx,
            'fg_pixels': fg, 'is_empty': is_empty,
            'class_pixels': count_class_pixels(mpatch, max(num_classes, 1)),
        })
    return meta

In [ ]:
# test: single-channel patches share image/mask names; multi-channel keeps _ch{i} channels
import tempfile
rng = np.random.default_rng(0)
with tempfile.TemporaryDirectory() as d:
    d = Path(d)

    # --- single-channel layout: image and mask filenames must match ---
    (d / 'single' / 'images').mkdir(parents=True)
    (d / 'single' / 'masks').mkdir(parents=True)
    img = rng.integers(0, 256, (40, 40), np.uint8)
    mask = np.zeros((40, 40), np.uint8); mask[10:30, 10:30] = 255
    cv2.imwrite(str(d / 'single' / 'images' / 'a.png'), img)
    cv2.imwrite(str(d / 'single' / 'masks' / 'a.png'), mask)
    cfg = PatchConfig(patch_size=20, stride=20, min_fg_pixels=1, max_empty_fraction=0.0)
    meta = extract_patches_from_sample('a', [d / 'single' / 'images' / 'a.png'], d / 'single' / 'masks' / 'a.png', d / 'single_out', cfg)
    assert len(meta) >= 1
    img_names = sorted(p.name for p in (d / 'single_out' / 'images').glob('*.png'))
    mask_names = sorted(p.name for p in (d / 'single_out' / 'masks').glob('*.png'))
    assert img_names == mask_names, f'mismatch: {img_names} vs {mask_names}'
    assert not any('_ch0' in n or '_mask' in n for n in img_names)
    assert all(m['fg_pixels'] >= 1 for m in meta)

    # --- multi-channel layout: channels keep _ch{i} suffixes ---
    (d / 'multi' / 'in').mkdir(parents=True)
    ch0 = rng.integers(0, 256, (40, 40), np.uint8)
    ch1 = rng.integers(0, 256, (40, 40), np.uint8)
    cv2.imwrite(str(d / 'multi' / 'in' / 'b_ch0.png'), ch0)
    cv2.imwrite(str(d / 'multi' / 'in' / 'b_ch1.png'), ch1)
    cv2.imwrite(str(d / 'multi' / 'in' / 'b_mask.png'), mask)
    meta2 = extract_patches_from_sample('b', [d / 'multi' / 'in' / 'b_ch0.png', d / 'multi' / 'in' / 'b_ch1.png'], d / 'multi' / 'in' / 'b_mask.png', d / 'multi_out', cfg)
    assert len(meta2) >= 1
    assert len(list((d / 'multi_out' / 'images').glob('*_ch0.png'))) == len(meta2)
    assert len(list((d / 'multi_out' / 'images').glob('*_ch1.png'))) == len(meta2)
    assert len(list((d / 'multi_out' / 'masks').glob('*.png'))) == len(meta2)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()